# 结果浏览入口（color_analyse_0727 v2）

这个 notebook 用于浏览 `result/final_analysis_<日期>/` 下的分析结果。

使用方法：依次运行单元格，在“过滤条件”单元格里修改 `WINDOW`、`SIGNAL`、`SUBJECT` 后重新运行下面的浏览单元格。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

# 自动定位结果目录：优先当前目录下的 result，其次上级目录
candidates = [Path.cwd(), Path.cwd().parent]
result_root = None
for cand in candidates:
    r = cand / 'result'
    if r.exists():
        result_root = r
        break
if result_root is None:
    raise FileNotFoundError('未找到 result 目录，请确认 notebook 位于 color_analyse_0727 或其 notebooks 子目录')

RESULT_DIR = sorted(result_root.glob('final_analysis_*'))[-1]
print('结果目录:', RESULT_DIR)

In [ ]:
# 输出索引总览
index = pd.read_csv(RESULT_DIR / 'output_index.csv')
print('输出文件总数:', len(index))
display(index.groupby(['stage', 'window', 'signal']).size().to_frame('n_files'))

In [ ]:
# ========== 过滤条件 ==========
WINDOW = '0-300'    # '0-300' 或 '100-400'
SIGNAL = 'lf'       # 'lf' 或 'broadband'
SUBJECT = ''        # 例如 'test001'，留空表示全部被试
# ==============================

In [ ]:
def load_stage(stage):
    frames = []
    for p in sorted((RESULT_DIR / stage).glob('*.csv')):
        df = pd.read_csv(p)
        if 'window' in df.columns:
            df = df[df.window == WINDOW]
        if 'signal' in df.columns:
            df = df[df.signal == SIGNAL]
        if SUBJECT and 'subject' in df.columns:
            df = df[df.subject == SUBJECT]
        frames.append((p.name, df))
    return frames

for name, df in load_stage('stage01_selection'):
    display(Markdown('### ' + name + '（共 ' + str(len(df)) + ' 行）'))
    display(df.head(30))

In [ ]:
# 浏览各阶段表格（解码、幅度/频谱、亮度）
for stage in ('stage02_amplitude_spectral', 'stage03_decoding', 'stage04_luminance'):
    for name, df in load_stage(stage):
        display(Markdown('### ' + stage + ' / ' + name + '（共 ' + str(len(df)) + ' 行）'))
        display(df.head(20))

In [ ]:
# 浏览变体专属图（按阶段）
for stage in ('stage01_selection', 'stage02_amplitude_spectral', 'stage03_decoding'):
    fig_dir = RESULT_DIR / stage / 'figures'
    if not fig_dir.exists():
        continue
    figs = sorted(fig_dir.glob('*_' + WINDOW + '_' + SIGNAL + '.png'))
    display(Markdown('### ' + stage + '（' + str(len(figs)) + ' 张图）'))
    for f in figs[:8]:
        display(Image(filename=str(f)))